# Train the waste sorting classifier on a Kaggle GPU

Runs the project's own training script, unchanged, from the public repository, for two architectures (ResNet18 and EfficientNet-B0).

**Before running:** in the notebook's right-hand panel, set **Accelerator → GPU** (T4 or P100) and turn **Internet on**. Both need a phone-verified Kaggle account.

**After running:** download `run.zip` from the **Output** panel and unzip it into the project's `artifacts/` folder.

In [ ]:
# 1. A GPU must be available; otherwise stop here rather than train for hours on a CPU.
import sys

import torch

assert torch.cuda.is_available(), "No GPU: set Accelerator -> GPU in the right-hand panel"
print("Python", sys.version.split()[0], "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2. The code, from GitHub. Cloned to /tmp so the dataset is not saved as notebook output.
REPO = "https://github.com/vladzpop11/waste-sorting-classifier.git"
!rm -rf /tmp/repo && git clone -q --depth 1 {REPO} /tmp/repo
%cd /tmp/repo
!git log -1 --format='commit %H (%s)'

In [ ]:
# 3. Kaggle already has PyTorch, torchvision, scikit-learn, pandas, Pillow and PyYAML.
#    The package itself is not installed: Kaggle runs Python 3.13, which the Intel-Mac pins
#    in pyproject.toml exclude, so the code runs from source (PYTHONPATH=src) instead.
!pip install -q "mlflow-skinny>=3" "sqlalchemy>=2" "alembic>=1.13"

In [ ]:
# 4. Download TrashNet and check it: archive SHA-256, published class counts, then the
#    training script checks every image against the committed split.
!PYTHONPATH=src python -m waste_sorting_classifier.data

In [ ]:
# 5. Train both candidate architectures (early stopping on validation loss). The model is
#    chosen later on the validation split only: the higher macro F1 wins, and within 0.01
#    the faster one (EfficientNet-B0 runs about 2x faster on CPU).
import os
import subprocess

for arch in ["resnet18", "efficientnet_b0"]:
    cmd = [sys.executable, "-m", "waste_sorting_classifier.train", "--arch", arch,
           "--output-dir", f"/kaggle/working/run/{arch}"]  # fmt: skip
    with subprocess.Popen(cmd, env={**os.environ, "PYTHONPATH": "src"}, text=True,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT) as proc:  # fmt: skip
        for line in proc.stdout:
            print(line, end="")
    assert proc.returncode == 0, f"training {arch} failed"

In [ ]:
# 6. One file to download: run.zip (per architecture: model.pt, metrics, history,
#    confusion matrix, MLflow store).
!git -C /tmp/repo rev-parse HEAD > /kaggle/working/run/commit.txt
!cd /kaggle/working && rm -f run.zip && zip -qr run.zip run
!ls -laR /kaggle/working/run | head -40 && ls -la /kaggle/working/run.zip